In [ ]:



import pandas as pd
import requests
from io import StringIO


url = "https://www.slickcharts.com/sp500"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)

tables = pd.read_html(StringIO(resp.text))
df = tables[0]


tickers = df["Symbol"].str.replace(".", "-", regex=False).tolist()


tickers = tickers[:190]

print(f"Got {len(tickers)} tickers")
print("Sample tickers:", tickers[:10])


pd.DataFrame({"ticker": tickers}).to_csv("./data/sp500_top190_tickers.csv", index=False)
print("Saved sp500_top190_tickers.csv")
#this cell aint gonna work , just use the direct csv file i have put under this path


In [1]:
#the yfinance part 
import yfinance as yf
import pandas as pd
from datetime import datetime


tickers = pd.read_csv("./data/sp500_top190_tickers.csv")["ticker"].tolist()
print(f"Using {len(tickers)} tickers")


start = "2010-01-01"
end = datetime.now().strftime("%Y-%m-%d")
print(f"Downloading Adj Close from {start} to {end}")


adj_close = pd.DataFrame()

batch_size = 40 
for i in range(0, len(tickers), batch_size):
    batch = tickers[i:i+batch_size]
    print(f"Batch {i//batch_size + 1}: {batch[0]} ... {batch[-1]}")

    
    data = yf.download(
        batch,
        start=start,
        end=end,
        progress=False,
        auto_adjust=False  
    )

    
    if isinstance(data.columns, pd.MultiIndex):
        batch_adj = data["Adj Close"]
    else:
        
        batch_adj = data[["Adj Close"]]
        batch_adj.columns = batch 

    
    adj_close = pd.concat([adj_close, batch_adj], axis=1)


adj_close = adj_close[tickers]

print("Raw Adj Close shape:", adj_close.shape)
print("First dates:", adj_close.index[:3])
print("Last dates:", adj_close.index[-3:])


Using 190 tickers
Batch 1: NVDA ... TMUS
Batch 2: IBM ... ETN
Batch 3: PANW ... TT
Batch 4: WDC ... CL
Batch 5: CI ... AZO
Raw Adj Close shape: (4088, 190)
First dates: DatetimeIndex(['2010-01-04', '2010-01-05', '2010-01-06'], dtype='datetime64[s]', name='Date', freq=None)
Last dates: DatetimeIndex(['2026-04-01', '2026-04-02', '2026-04-06'], dtype='datetime64[s]', name='Date', freq=None)


In [2]:

adj_close.to_csv('./data/sp500_190_16y_adjclose_raw.csv')
print("Saved sp500_190_16y_adjclose_raw.csv (4k days × 190 stocks)")
print("Shape:", adj_close.shape)
print("NaN count:", adj_close.isna().sum().sum())
print("READY FOR SCREENING!")


Saved sp500_190_16y_adjclose_raw.csv (4k days × 190 stocks)
Shape: (4088, 190)
NaN count: 39533
READY FOR SCREENING!


In [3]:

import pandas as pd
df = pd.read_csv('./data/sp500_190_16y_adjclose_raw.csv', index_col=0, parse_dates=True)
df_filled = df.ffill()
df_filled.to_csv('./data/sp500_190_16y_adjclose_ffill.csv')
print("ffill complete. File saved.")


import itertools
tickers = df_filled.columns.tolist()
all_pairs = list(itertools.combinations(tickers, 2))
print(f"{len(all_pairs):,} pairs formed: {all_pairs[:5]}...")


pairs_df = pd.DataFrame(all_pairs, columns=['stock1', 'stock2'])
pairs_df.to_csv('data/sp500_17955_pairs.csv', index=False)
print("Saved: s&p500_17955_pairs.csv")
print("SPEC FULLY COMPLETE - Ready for Engle-Granger screening!")


ffill complete. File saved.
17,955 pairs formed: [('NVDA', 'AAPL'), ('NVDA', 'MSFT'), ('NVDA', 'AMZN'), ('NVDA', 'GOOGL'), ('NVDA', 'GOOG')]...
Saved: s&p500_17955_pairs.csv
SPEC FULLY COMPLETE - Ready for Engle-Granger screening!
